# Filtering out condition-related words

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [ ]:
# Load .csv file
df = pd.read_csv('../data/unfiltered/concatenated/author_documents_full.csv')

In [ ]:
# How many authors mention the words

features = ['stomach', 'sick', 'constipation', # top IBS-related features 
            'cyst', # top endometriosis-related feature
            # General symptoms and conditions related to both endometriosis and IBS
            'bloating', 'diarrhoea', 'diahrrea', 'pelvic pain', 'abdominal pain', 
            'stomach pain','painful periods', 'dysmenorrhea', 'nausea', 'vomiting', 'fatigue', 
              'infertility', 'cramps']

for feature in features:
    if feature == 'diahrrea':  # Handle common misspelling
       feature = 'diarrhoea'
    count = df['concatenated_text'].str.contains(feature, case=False, na=False).sum()
    print(f"The word '{feature}' is mentioned by {count} authors.")
    class_distribution = df[df['concatenated_text'].str.contains(feature, case=False, na=False)]['label'].value_counts()
    print(f"Class distribution for '{feature}':\n{class_distribution}\n")
    percentage = (class_distribution / class_distribution.sum()) * 100
    print(f"Percentage distribution for '{feature}':\n{percentage}\n")

In [ ]:
# Plotting the distribution of features across classes
counts = {}
for feature in features:
    if feature == 'diahrrea':  # Handle common misspelling
       feature = 'diarrhoea'
    mask = df['concatenated_text'].str.contains(feature, case=False, na=False)
    counts[feature] = df[mask]['label'].value_counts()

count_df = pd.DataFrame(counts).T.fillna(0)

width = 0.3

ax = count_df.plot(kind='bar', figsize=(10, 5))
plt.title('Distribution of features across classes')
plt.xlabel('Feature')
plt.ylabel('Count')
plt.xticks(rotation=70)
plt.legend(title='Class')

row_totals = count_df.sum(axis=1).values

for container_idx, container in enumerate(ax.containers):
    for bar_idx, bar in enumerate(container):
        height = bar.get_height()
        row_total = row_totals[bar_idx]
        if row_total > 0:
            pct = None #f'{height / row_total * 100:.1f}%'
            ax.text(bar.get_x() + bar.get_width() / 3, height + 0.5, pct,
                    ha='center', va='bottom', fontsize=8)

#plt.tight_layout()
plt.show()

In [ ]:
# Filter out the words

filtered_df = df.copy()
for feature in features:
    filtered_df['concatenated_text'] = filtered_df['concatenated_text'].str.replace(
        r'\b' + feature + r'\b', '', flags=re.IGNORECASE, regex=True
    )
filtered_df.to_csv('../data/filtered/concatenated/author_documents_filtered.csv', index=False)